# Live Pipeline Status + Latest Feathers (DB-backed)
This dashboard reads pipeline state from `run_stats.sqlite3` under `data/runs/<run_id>/pipeline_metrics/`.


In [ ]:
import sqlite3
from pathlib import Path
from IPython.display import display, Markdown, Image
import ipywidgets as widgets

RUN_ID = ""

def detect_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/Users/openteams/Feather_Molt_Project')]
    for c in candidates:
        if (c / 'data' / 'runs').exists() and (c / 'src').exists():
            return c
    return Path.cwd()

REPO_ROOT = detect_repo_root()
RUNS_DIR = REPO_ROOT / 'data' / 'runs'

def auto_run_id() -> str:
    if not RUNS_DIR.exists():
        return ''
    run_dirs = [d for d in RUNS_DIR.iterdir() if d.is_dir()]
    if not run_dirs:
        return ''
    run_dirs.sort(key=lambda d: d.stat().st_mtime, reverse=True)
    return run_dirs[0].name

if not RUN_ID:
    RUN_ID = auto_run_id()

print('REPO_ROOT =', REPO_ROOT)
print('RUN_ID =', RUN_ID if RUN_ID else '<none found>')


In [ ]:
def _db_path(run_id: str) -> Path:
    return RUNS_DIR / run_id / 'pipeline_metrics' / 'run_stats.sqlite3'

def _run_output_dir(run_id: str) -> Path:
    return RUNS_DIR / run_id / 'processed'

def fetch_status(run_id: str) -> dict:
    if not run_id.strip():
        raise ValueError('No run id found. Start a run first, or set RUN_ID manually.')
    db_path = _db_path(run_id)
    if not db_path.exists():
        raise FileNotFoundError(f'No DB found: {db_path}')

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    total = dict(cur.execute('''
        SELECT
          COUNT(*) AS processed,
          SUM(CASE WHEN success=1 THEN 1 ELSE 0 END) AS success,
          SUM(CASE WHEN success=0 THEN 1 ELSE 0 END) AS failed,
          SUM(feathers_saved) AS feathers_saved,
          AVG(duration_ms) AS avg_ms_per_image,
          AVG(CASE WHEN vlm_score IS NOT NULL THEN vlm_score END) AS vlm_score_avg,
          SUM(CASE WHEN vlm_score IS NOT NULL THEN 1 ELSE 0 END) AS vlm_score_count,
          SUM(CASE WHEN metadata_source='vlm_fallback' THEN 1 ELSE 0 END) AS metadata_from_vlm_fallback,
          SUM(CASE WHEN bird_id != 'UNKNOWN' AND date_text != 'UNKNOWN' THEN 1 ELSE 0 END) AS metadata_full_known,
          SUM(CASE WHEN bird_id = 'UNKNOWN' OR date_text = 'UNKNOWN' THEN 1 ELSE 0 END) AS metadata_unknown_any,
          SUM(CASE WHEN vlm_all_feathers_covered=1 THEN 1 ELSE 0 END) AS vlm_all_feathers_covered_true,
          SUM(CASE WHEN vlm_all_feathers_covered IS NOT NULL THEN 1 ELSE 0 END) AS vlm_all_feathers_covered_count,
          SUM(CASE WHEN vlm_background_leakage_detected=1 THEN 1 ELSE 0 END) AS vlm_background_leakage_detected,
          SUM(CASE WHEN vlm_background_leakage_detected IS NOT NULL THEN 1 ELSE 0 END) AS vlm_background_leakage_count,
          SUM(CASE WHEN vlm_grouped_boxes_detected=1 THEN 1 ELSE 0 END) AS vlm_grouped_boxes_detected,
          SUM(CASE WHEN vlm_grouped_boxes_detected IS NOT NULL THEN 1 ELSE 0 END) AS vlm_grouped_boxes_count,
          SUM(retry_used) AS retry_used,
          SUM(retry_selected) AS retry_selected
        FROM image_stats
        WHERE run_id=?
    ''' , (run_id,)).fetchone())

    nodes = [dict(r) for r in cur.execute('''
        SELECT
          node_id,
          COUNT(*) AS processed,
          SUM(CASE WHEN success=1 THEN 1 ELSE 0 END) AS success,
          SUM(CASE WHEN success=0 THEN 1 ELSE 0 END) AS failed,
          SUM(feathers_saved) AS feathers_saved,
          AVG(duration_ms) AS avg_ms_per_image,
          AVG(CASE WHEN vlm_score IS NOT NULL THEN vlm_score END) AS vlm_score_avg,
          SUM(CASE WHEN vlm_all_feathers_covered=1 THEN 1 ELSE 0 END) AS vlm_all_feathers_covered_true,
          SUM(CASE WHEN vlm_all_feathers_covered IS NOT NULL THEN 1 ELSE 0 END) AS vlm_all_feathers_covered_count,
          SUM(CASE WHEN vlm_background_leakage_detected=1 THEN 1 ELSE 0 END) AS vlm_background_leakage_detected,
          SUM(CASE WHEN vlm_background_leakage_detected IS NOT NULL THEN 1 ELSE 0 END) AS vlm_background_leakage_count,
          SUM(CASE WHEN vlm_grouped_boxes_detected=1 THEN 1 ELSE 0 END) AS vlm_grouped_boxes_detected,
          SUM(CASE WHEN vlm_grouped_boxes_detected IS NOT NULL THEN 1 ELSE 0 END) AS vlm_grouped_boxes_count,
          SUM(retry_used) AS retry_used,
          SUM(retry_selected) AS retry_selected
        FROM image_stats
        WHERE run_id=?
        GROUP BY node_id
        ORDER BY node_id
    ''', (run_id,)).fetchall()]

    conn.close()
    return {'totals': total, 'nodes': nodes}

def _fmt_rate(num: int, den: int) -> str:
    if not den:
        return 'n/a'
    return f'{(num/den):.2%}'

def render_status(run_id: str):
    s = fetch_status(run_id)
    t = s['totals']
    processed = int(t.get('processed') or 0)
    success = int(t.get('success') or 0)
    failed = int(t.get('failed') or 0)
    feathers_saved = int(t.get('feathers_saved') or 0)
    avg_ms = float(t.get('avg_ms_per_image') or 0)

    lines = []
    lines.append(f"## Run `{run_id}`")
    lines.append(f"- processed rows: **{processed}**")
    lines.append(f"- success: **{success}**, failed: **{failed}**")
    lines.append(f"- feathers_saved: **{feathers_saved}**")
    lines.append(f"- avg_ms_per_image: **{avg_ms:.1f}**")
    lines.append(f"- vlm_score_avg: **{t.get('vlm_score_avg')}** (count={int(t.get('vlm_score_count') or 0)})")
    lines.append(f"- metadata full known: **{int(t.get('metadata_full_known') or 0)}**")
    lines.append(f"- metadata unknown any: **{int(t.get('metadata_unknown_any') or 0)}**")
    lines.append(f"- metadata via VLM fallback: **{int(t.get('metadata_from_vlm_fallback') or 0)}**")
    lines.append(f"- retry used: **{int(t.get('retry_used') or 0)}**")
    lines.append(f"- retry selected: **{int(t.get('retry_selected') or 0)}**")

    covered_true = int(t.get('vlm_all_feathers_covered_true') or 0)
    covered_count = int(t.get('vlm_all_feathers_covered_count') or 0)
    leakage_true = int(t.get('vlm_background_leakage_detected') or 0)
    leakage_count = int(t.get('vlm_background_leakage_count') or 0)
    grouped_true = int(t.get('vlm_grouped_boxes_detected') or 0)
    grouped_count = int(t.get('vlm_grouped_boxes_count') or 0)

    lines.append(f"- VLM all-feathers-covered: **{covered_true}/{covered_count}** ({_fmt_rate(covered_true, covered_count)})")
    lines.append(f"- VLM background leakage: **{leakage_true}/{leakage_count}** ({_fmt_rate(leakage_true, leakage_count)})")
    lines.append(f"- VLM grouped boxes: **{grouped_true}/{grouped_count}** ({_fmt_rate(grouped_true, grouped_count)})")

    lines.append('\n### Per-node (from DB)')
    for n in s['nodes']:
        lines.append(
            f"- `{n['node_id']}`: processed={int(n['processed'] or 0)} success={int(n['success'] or 0)} failed={int(n['failed'] or 0)} feathers={int(n['feathers_saved'] or 0)} avg_ms={float(n['avg_ms_per_image'] or 0):.1f} vlm_avg={n['vlm_score_avg']} retry_used={int(n['retry_used'] or 0)} retry_selected={int(n['retry_selected'] or 0)}"
        )
    display(Markdown('\n'.join(lines)))

render_status(RUN_ID)


In [ ]:
def render_latest_feathers(run_id: str, max_show: int = 12):
    db_path = _db_path(run_id)
    out_dir = _run_output_dir(run_id)
    if not db_path.exists():
        display(Markdown(f'No stats DB yet: `{db_path}`'))
        return

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    rows = cur.execute('''
        SELECT image_path, node_id, ts, feathers_saved, vlm_score, bird_id, date_text
        FROM image_stats
        WHERE run_id=? AND success=1
        ORDER BY ts DESC
        LIMIT ?
    ''', (run_id, max_show)).fetchall()
    conn.close()

    display(Markdown(f'### Latest Successful Sources ({len(rows)} rows)'))
    if not rows:
        return

    for r in rows:
        src_stem = Path(r['image_path']).stem
        bird = r['bird_id'] or 'UNKNOWN'
        date_text = r['date_text'] or 'UNKNOWN'
        pattern = f"{src_stem}_{bird}_{date_text}_Feather_*.jpg"
        segs = sorted(out_dir.glob(pattern))[:5]
        hdr = f"- `{src_stem}` node={r['node_id']} ts={r['ts']} feathers_saved={r['feathers_saved']} vlm_score={r['vlm_score']}"
        display(Markdown(hdr))
        if not segs:
            display(Markdown('  - No segment files resolved from DB row metadata yet.'))
            continue
        for s in segs:
            display(Image(filename=str(s), width=240))

render_latest_feathers(RUN_ID, max_show=10)


In [ ]:
# Problematic Images + Segments from SQLite
def render_problematic_segments(run_id: str, limit: int = 12):
    if not run_id.strip():
        raise ValueError('RUN_ID is empty')
    db_path = _db_path(run_id)
    out_dir = _run_output_dir(run_id)
    if not db_path.exists():
        display(Markdown(f'No stats DB yet: `{db_path}`'))
        return

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    rows = cur.execute('''
        SELECT image_path, vlm_score, vlm_grouped_boxes_detected,
               vlm_background_leakage_detected, vlm_all_feathers_covered,
               vlm_notes, retry_used, retry_selected, feathers_saved, metadata_source, bird_id, date_text
        FROM image_stats
        WHERE run_id = ?
        ORDER BY
          CASE WHEN vlm_background_leakage_detected = 1 THEN 0 ELSE 1 END,
          CASE WHEN vlm_all_feathers_covered = 0 THEN 0 ELSE 1 END,
          CASE WHEN feathers_saved <= 2 THEN 0 ELSE 1 END,
          CASE WHEN vlm_grouped_boxes_detected = 1 THEN 0 ELSE 1 END,
          CASE WHEN vlm_score IS NULL THEN 1 ELSE 0 END,
          vlm_score ASC,
          feathers_saved ASC
        LIMIT ?
    ''', (run_id, limit)).fetchall()
    conn.close()

    if not rows:
        display(Markdown('No rows in stats DB yet.'))
        return

    display(Markdown(f'### Problematic Samples ({len(rows)})'))
    for r in rows:
        stem = Path(r['image_path']).stem
        bird = r['bird_id'] or 'UNKNOWN'
        date_text = r['date_text'] or 'UNKNOWN'
        segs = sorted(out_dir.glob(f"{stem}_{bird}_{date_text}_Feather_*.jpg"))[:6]

        summary = (
            f"- `{Path(r['image_path']).name}`  "
            f"vlm_score={r['vlm_score']} grouped={r['vlm_grouped_boxes_detected']} leakage={r['vlm_background_leakage_detected']} covered={r['vlm_all_feathers_covered']} "
            f"retry_used={r['retry_used']} retry_selected={r['retry_selected']} feathers_saved={r['feathers_saved']} "
            f"metadata={r['metadata_source']} bird={bird} date={date_text}"
        )
        display(Markdown(summary))
        if r['vlm_notes']:
            display(Markdown(f"  - VLM notes: {r['vlm_notes']}"))

        bbox = out_dir / f"{stem}_BoundingBoxes.jpg"
        if bbox.exists():
            display(Image(filename=str(bbox), width=300))

        if not segs:
            display(Markdown('  - No segment files yet for this source image.'))
            continue
        for s in segs:
            display(Image(filename=str(s), width=230))

render_problematic_segments(RUN_ID, limit=10)


In [ ]:
widgets.interact(lambda rid: render_status(rid), rid=widgets.Text(value=RUN_ID, description='RUN_ID:'))
widgets.interact(lambda rid: render_problematic_segments(rid, limit=12), rid=widgets.Text(value=RUN_ID, description='Problem RUN_ID:'))
